# Scraping Wikipedia Animals (https://en.wikipedia.org/wiki/List_of_animal_names)

In [1]:
import cloudscraper
from bs4 import BeautifulSoup

url = "https://en.wikipedia.org/wiki/List_of_animal_names"

scraper = cloudscraper.create_scraper()
response = scraper.get(url)

print(response.status_code)

soup = BeautifulSoup(response.text, "html.parser")

# Comprobar si se ha cargado bien el html de la página
soup.title


200


<title>List of animal names - Wikipedia</title>

In [2]:
table = soup.find_all("table")[2]
table

<table class="wikitable sortable sticky-header">
<tbody><tr>
<th>Animal</th>
<th>Young</th>
<th><a href="/wiki/Female" title="Female">Female</a></th>
<th><a href="/wiki/Male" title="Male">Male</a></th>
<th><a href="/wiki/Collective_noun" title="Collective noun">Collective noun</a></th>
<th><a href="/wiki/Collateral_adjective" title="Collateral adjective">Collateral adjective</a></th>
<th>Culinary noun for <a href="/wiki/Meat" title="Meat">meat</a>
</th></tr>
<tr>
<th colspan="7"><span class="anchor" id="A"></span><b>A</b>
</th></tr>
<tr>
<td><a href="/wiki/Albatross" title="Albatross">Albatross</a></td>
<td>chick</td>
<td class="table-na" data-sort-value="" style="background: var(--background-color-interactive, #ececec); color: var(--color-base, inherit); vertical-align: middle; text-align: center;"><span aria-hidden="true">—</span><link href="mw-data:TemplateStyles:r1152813436" rel="mw-deduplicated-inline-style"><span class="sr-only">N/a</span></link></td>
<td class="table-na" data-so

In [3]:
wikipedia_animals = []

for tr in table.select("tbody > tr"):
    animal = tr.select_one("td")
    if animal is not None:
        wikipedia_animals.append(animal.select_one("a").get("title"))

In [4]:
print(f"{len(wikipedia_animals)} animales encontrados")
print(wikipedia_animals)


183 animales encontrados
['Albatross', 'Alligator', 'Alpaca', 'Ant', 'Antelope', 'Ape', 'Armadillo', 'Donkey', 'Baboon', 'Badger', 'Bat', 'Bear', 'Beaver', 'Bee', 'Beetle', 'Binturong', 'Bison', 'Wild boar', 'Bobcat', 'Bull', 'Butterfly', 'Camel', 'Northern cardinal', 'Cat', 'Cattle', 'Cheetah', 'Chicken', 'Chinchilla', 'Chough', 'Cobra', 'Cockroach', 'Cod', 'Cormorant', 'Cow', 'Crab', 'Crane (bird)', 'Crocodile', 'Crow', 'Deer', 'Dog', 'Squalidae', 'Dolphin', 'Donkey', 'Dove', 'Duck', 'Dunlin', 'Eagle', 'Echidna', 'Eel', 'Elephant', 'Elk', 'Emu', 'Falcon', 'Ferret', 'Finch', 'Flamingo', 'Fly', 'Fox', 'Frog', 'Gecko', 'Gerbil', 'Giant panda', 'Giraffe', 'Gnat', 'Wildebeest', 'Goat', 'European goldfinch', 'Common merganser', 'Goose', 'Gorilla', 'Goshawk', 'Grasshopper', 'Grouse', 'Guanaco', 'Guinea fowl', 'Guinea pig', 'Hamster', 'Hare', 'Hawk', 'Hedgehog', 'Heron', 'Herring', 'Hippopotamus', 'Hornet', 'Horse', 'Human', 'Hyena', 'Jaguar', 'Jay', 'Jellyfish', 'Junglefowl', 'Kangaroo', 'K

In [ ]:
import wikipediaapi

def get_markdown(animal):

    irrelevant_sections = ["see also", "references", "sources", "external links", "notes"]

    wiki = wikipediaapi.Wikipedia(
        user_agent='MyProjectName (merlin@example.com)',
        language='en',
        extract_format=wikipediaapi.ExtractFormat.WIKI
    )

    page = wiki.page(animal)
    if not page.exists():
        return "Página no encontrada"
    
    # Main title
    md_content = [f"# {page.title}", f"{page.summary}\n"]

    # Recursively process sections
    def process_sections(sections, level=2):
        for s in sections:
            if s.title.lower() in irrelevant_sections:
                return
            
            header = "#" * level
            md_content.append(f"{header} {s.title}")
            if s.text:
                md_content.append(f"{s.text}\n")
            
            # Process subsections
            if s.sections:
                process_sections(s.sections, level + 1)

    process_sections(page.sections)
    
    return "\n".join(md_content)



In [ ]:
# Example
markdown_text = get_markdown("Spider")

with open("../data/spider.md", 'w') as file:
    file.write(markdown_text)

In [8]:
for animal in wikipedia_animals:
    markdown_text = get_markdown(animal)
    with open(f"../data/{animal.lower().replace(" ", "_")}.md", 'w') as file:
        file.write(markdown_text)